# B.Tech Probability & Statistics Project
## Time-Series Analysis and Forecasting of AI Model Training Compute Using Real-World Data
### Notebook 02: Statistical Modeling, Extended 7-Fold Backtesting, and Final Model Selection

**Academic Context & Syllabus Alignment:**
- **Module I:** Descriptive statistics, frequency distributions, measures of central tendency (mean, median), measures of dispersion (variance, standard deviation, IQR), scatter plots, and histograms.
- **Module VII:** Hypothesis testing, null and alternate hypotheses, Welch's two-sample t-test for difference in means with unequal variances, degrees of freedom, and p-values.
- **Module IX:** Bivariate correlation, covariance, Pearson product-moment correlation coefficient $r$, Spearman rank correlation coefficient $\rho$.
- **Module X:** Simple linear regression, polynomial (quadratic) regression, coefficient of determination ($R^2$), adjusted $R^2$, standard error, residual diagnostics, and Durbin-Watson autocorrelation testing.
- **Time-Series Forecasting & Model Selection:** Extended rolling-origin backtesting (2019–2025, 7 folds), Holt’s Linear Trend (Exponential Smoothing), ARIMA specifications, out-of-sample evaluation ($R^2_{\text{OOS}}$, MAE, RMSE), and multi-step extrapolation stability analysis through 2030.

**Research Goal:** Systematically compare six candidate forecasting models using rigorous out-of-sample backtesting, evaluate multi-year extrapolation stability, validate against partial 2026 data, and select the primary forecasting model for 2027–2030 projections.

### 1. Environment & Setup
We import scientific libraries (`pandas`, `numpy`, `matplotlib.pyplot`, `scipy.stats`, `statsmodels`) and configure plotting styles.

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.holtwinters import Holt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# Configure high-quality publication aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

for d in ['outputs', 'graphs', 'data']:
    os.makedirs(f'../{d}' if os.path.exists('../outputs') else d, exist_ok=True)

print('Environment and libraries successfully configured.')

Environment and libraries successfully configured.


### 2. Task 1: Load Yearly Compute Dataset (2010–2026)
We load `outputs/yearly_ai_compute.csv` and inspect the primary target:
$$Y_t = \text{median}\left(\log_{10}(\text{Training compute (FLOP)})\right) \quad \text{for year } t$$
We display the full table from 2010 to 2026, explicitly noting that 2026 is a partial year (recorded through September 21, 2026).

In [4]:
yac_path = 'outputs/yearly_ai_compute.csv' if os.path.exists('outputs/yearly_ai_compute.csv') else '../outputs/yearly_ai_compute.csv'
yearly_df = pd.read_csv(yac_path)

yearly_subset = yearly_df[(yearly_df['Year'] >= 2010) & (yearly_df['Year'] <= 2026)].copy()
yearly_subset['Status'] = 'Full Year'
yearly_subset.loc[yearly_subset['Year'] == 2026, 'Status'] = 'PARTIAL (through Sep 21)'

display_cols = ['Year', 'models_with_valid_compute', 'mean_training_compute', 
                'median_training_compute', 'mean_log10_compute', 'median_log10_compute', 'Status']
yearly_subset[display_cols]

### 3. Task 2: Module IX: Correlation Analysis
We test the strength and statistical significance of the linear and monotonic relationship between calendar year ($X$) and median $\log_{10}(\text{compute})$ ($Y$) over the historical training horizon (2010–2025, $N=16$).

In [6]:
train_df = yearly_subset[yearly_subset['Year'] <= 2025].copy()
x_train = train_df['Year'].values
y_train = train_df['median_log10_compute'].values
n_train = len(x_train)

pearson_r, pearson_p = stats.pearsonr(x_train, y_train)
spearman_rho, spearman_p = stats.spearmanr(x_train, y_train)

print('=== MODULE IX: CORRELATION ANALYSIS ===')
print(f'Sample Size (N):              {n_train}')
print(f'Pearson Correlation (r):      {pearson_r:.6f} (p-value = {pearson_p:.4e})')
print(f'Spearman Rank Correlation (ρ): {spearman_rho:.6f} (p-value = {spearman_p:.4e})')
print('Disclaimer: Correlation confirms empirical co-movement over time and does NOT imply causation.')

=== MODULE IX: CORRELATION ANALYSIS ===
Sample Size (N):              16
Pearson Correlation (r):      0.969065 (p-value = 6.6992e-10)
Spearman Rank Correlation (ρ): 0.970588 (p-value = 4.7234e-10)
Disclaimer: Correlation confirms empirical co-movement over time and does NOT imply causation.


### 4. Task 3 & 4: Module X: Trend Regressions (Linear & Quadratic)
We fit both the simple linear model and the centered quadratic polynomial model:
- Linear: $Y_t = \beta_0 + \beta_1 t + \epsilon_t$
- Quadratic: $Y_t = \beta_0 + \beta_1 (t - 2010) + \beta_2 (t - 2010)^2 + \epsilon_t$

In [8]:
# 1. Linear OLS
slope_lin, intercept_lin, r_val_lin, p_val_lin, se_slope_lin = stats.linregress(x_train, y_train)
r2_lin = r_val_lin**2
adj_r2_lin = 1 - (1 - r2_lin) * (n_train - 1) / (n_train - 2)
growth_factor_lin = 10**slope_lin
doubling_time_years = np.log10(2) / slope_lin

print('=== MODULE X: SIMPLE LINEAR REGRESSION ===')
print(f'Fitted Equation:       Y_t = {intercept_lin:.4f} + {slope_lin:.4f} · Year')
print(f'R-squared:             {r2_lin:.4f} (Adjusted R2 = {adj_r2_lin:.4f})')
print(f'Annual Growth Factor:  10^(beta1) = {growth_factor_lin:.2f}x per year')
print(f'Doubling Time:         {doubling_time_years:.2f} years ({doubling_time_years*12:.1f} months)')

# 2. Quadratic Polynomial
t_c = x_train - 2010
p2_coeffs = np.polyfit(t_c, y_train, 2)
y_pred_q_train = np.polyval(p2_coeffs, t_c)
ss_res_q = np.sum((y_train - y_pred_q_train)**2)
ss_tot_train = np.sum((y_train - np.mean(y_train))**2)
r2_quad = 1 - ss_res_q / ss_tot_train
adj_r2_quad = 1 - (1 - r2_quad) * (n_train - 1) / (n_train - 3)

print('\n=== MODULE X: QUADRATIC REGRESSION ===')
print(f'Coefficients:          beta0={p2_coeffs[2]:.4f}, beta1={p2_coeffs[1]:.4f}, beta2={p2_coeffs[0]:.6f}')
print(f'R-squared:             {r2_quad:.4f} (Adjusted R2 = {adj_r2_quad:.4f})')

=== MODULE X: SIMPLE LINEAR REGRESSION ===
Fitted Equation:       Y_t = -1156.0523 + 0.5828 · Year
R-squared:             0.9391 (Adjusted R2 = 0.9347)
Annual Growth Factor:  10^(beta1) = 3.83x per year
Doubling Time:         0.52 years (6.2 months)

=== MODULE X: QUADRATIC REGRESSION ===
Coefficients:          beta0=15.9728, beta1=0.3052, beta2=0.018502
R-squared:             0.9550 (Adjusted R2 = 0.9481)


### 5. Extended 7-Fold Rolling-Origin Backtesting (2019–2025)
We test all candidate models across 7 consecutive out-of-sample evaluation folds without lookahead bias:
- Train 2010–2018 $\rightarrow$ Test 2019
- Train 2010–2019 $\rightarrow$ Test 2020
- Train 2010–2020 $\rightarrow$ Test 2021
- Train 2010–2021 $\rightarrow$ Test 2022
- Train 2010–2022 $\rightarrow$ Test 2023
- Train 2010–2023 $\rightarrow$ Test 2024
- Train 2010–2024 $\rightarrow$ Test 2025

In [10]:
test_years = list(range(2019, 2026)) # 7 folds
models_to_test = [
    'Naive', 'Linear Trend', 'Quadratic Trend', 
    'Holt Linear Trend', 'ARIMA(0,1,0)+drift', 'ARIMA(1,1,0)+drift'
]

backtest_7f_records = []
yearly_series = yearly_subset.set_index('Year')['median_log10_compute']

for ty in test_years:
    train_sub = yearly_series.loc[2010:ty-1]
    y_act = yearly_series.loc[ty]
    
    # 1. Naive
    p_naive = train_sub.iloc[-1]
    
    # 2. Linear
    s_tr, int_tr, _, _, _ = stats.linregress(train_sub.index.values, train_sub.values)
    p_lin = int_tr + s_tr * ty
    
    # 3. Quadratic
    tc_tr = train_sub.index.values - 2010
    tc_val = ty - 2010
    q_poly = np.polyfit(tc_tr, train_sub.values, 2)
    p_quad = np.polyval(q_poly, tc_val)
    
    # 4. Holt Linear Trend
    h_fit = Holt(train_sub.values, initialization_method='estimated').fit()
    p_holt = h_fit.forecast(1)[0]
    
    # 5. ARIMA(0,1,0) with drift
    ar010 = ARIMA(train_sub.values, order=(0,1,0), trend='t').fit()
    p_ar010 = ar010.forecast(1)[0]
    
    # 6. ARIMA(1,1,0) with drift
    ar110 = ARIMA(train_sub.values, order=(1,1,0), trend='t').fit()
    p_ar110 = ar110.forecast(1)[0]
    
    preds_dict = {
        'Naive': p_naive,
        'Linear Trend': p_lin,
        'Quadratic Trend': p_quad,
        'Holt Linear Trend': p_holt,
        'ARIMA(0,1,0)+drift': p_ar010,
        'ARIMA(1,1,0)+drift': p_ar110
    }
    
    for m_name in models_to_test:
        pred_val = preds_dict[m_name]
        err = y_act - pred_val
        backtest_7f_records.append({
            'Model': m_name,
            'Test_Year': ty,
            'Actual_Y_t': y_act,
            'Predicted_Y_t': pred_val,
            'Error': err,
            'Absolute_Error': abs(err),
            'Squared_Error': err**2
        })

backtest_7f_df = pd.DataFrame(backtest_7f_records)
out_bt = 'outputs/backtest_results.csv' if os.path.exists('outputs') else '../outputs/backtest_results.csv'
backtest_7f_df.to_csv(out_bt, index=False)

# Metrics calculation
y_act_vals = backtest_7f_df[backtest_7f_df['Model'] == 'Naive']['Actual_Y_t'].values
ss_tot_bt = np.sum((y_act_vals - np.mean(y_act_vals))**2)

comparison_rows = []
for m_name in models_to_test:
    sub = backtest_7f_df[backtest_7f_df['Model'] == m_name]
    mae = sub['Absolute_Error'].mean()
    rmse = np.sqrt(sub['Squared_Error'].mean())
    ss_res = sub['Squared_Error'].sum()
    oos_r2 = 1 - ss_res / ss_tot_bt
    
    comparison_rows.append({
        'Model': m_name,
        'Backtest_MAE': round(mae, 4),
        'Backtest_RMSE': round(rmse, 4),
        'OOS_R2': round(oos_r2, 4)
    })

comp_df = pd.DataFrame(comparison_rows)
print('=== 7-FOLD ROLLING BACKTEST METRICS (2019–2025) ===')
print(comp_df.to_string(index=False))
print('\nKey Finding: Quadratic model collapsed on 7-fold test (RMSE=1.209, OOS R2=0.413);')
print('Linear Trend maintained steady RMSE=0.890 and OOS R2=0.682; ARIMA(0,1,0) achieved lowest RMSE=0.824.')

=== 7-FOLD ROLLING BACKTEST METRICS (2019–2025) ===
             Model  Backtest_MAE  Backtest_RMSE  OOS_R2
             Naive        0.9210         1.1438  0.4744
      Linear Trend        0.8023         0.8900  0.6818
   Quadratic Trend        0.9454         1.2088  0.4130
 Holt Linear Trend        0.8481         1.0035  0.5955
ARIMA(0,1,0)+drift        0.6701         0.8241  0.7272
ARIMA(1,1,0)+drift        0.7083         0.8407  0.7161

Key Finding: Quadratic model collapsed on 7-fold test (RMSE=1.209, OOS R2=0.413);
Linear Trend maintained steady RMSE=0.890 and OOS R2=0.682; ARIMA(0,1,0) achieved lowest RMSE=0.824.


### 6. Multi-Step Extrapolation Stability & 2026 Snapshot Validation
We compare candidate models on the 2026 partial-year snapshot ($Y_{2026} = 24.6355$) and inspect their multi-year extrapolations through 2030.

In [12]:
actual_2026 = yearly_subset.loc[yearly_subset['Year'] == 2026, 'median_log10_compute'].values[0]

# Full fits on 2010-2025
s_full, int_full, _, _, se_full = stats.linregress(x_train, y_train)
p2_full = np.polyfit(x_train - 2010, y_train, 2)
holt_full = Holt(y_train, initialization_method='estimated').fit()
ar010_full = ARIMA(y_train, order=(0,1,0), trend='t').fit()

p2026_lin = int_full + s_full * 2026
p2026_quad = np.polyval(p2_full, 2026 - 2010)
p2026_holt = holt_full.forecast(1)[0]
p2026_ar010 = ar010_full.forecast(1)[0]

print('=== 2026 PARTIAL-YEAR SNAPSHOT VALIDATION ===')
print(f'Actual 2026 Median (through Sep 21): {actual_2026:.4f}')
print(f'Linear Trend Prediction:             {p2026_lin:.4f} (Error = {actual_2026 - p2026_lin:+.4f})')
print(f'Holt Linear Trend Prediction:        {p2026_holt:.4f} (Error = {actual_2026 - p2026_holt:+.4f})')
print(f'ARIMA(0,1,0)+drift Prediction:       {p2026_ar010:.4f} (Error = {actual_2026 - p2026_ar010:+.4f})')
print(f'Quadratic Trend Prediction:          {p2026_quad:.4f} (Error = {actual_2026 - p2026_quad:+.4f})')

print('\n=== 2030 EXTRAPOLATION COMPARISON ===')
print(f'Linear Trend 2030:    10^{int_full + s_full * 2030:.2f} = {10**(int_full + s_full * 2030):.2e} FLOP (Physically plausible)')
print(f'Holt Linear 2030:     10^{holt_full.forecast(5)[-1]:.2f} = {10**(holt_full.forecast(5)[-1]):.2e} FLOP (Dynamic trend)')
print(f'ARIMA(0,1,0) 2030:    10^{ar010_full.forecast(5)[-1]:.2f} = {10**(ar010_full.forecast(5)[-1]):.2e} FLOP (Differenced drift)')
print(f'Quadratic Trend 2030: 10^{np.polyval(p2_full, 20):.2f} = {10**(np.polyval(p2_full, 20)):.2e} FLOP (Explosive/implausible)')

=== 2026 PARTIAL-YEAR SNAPSHOT VALIDATION ===
Actual 2026 Median (through Sep 21): 24.6355
Linear Trend Prediction:             24.6496 (Error = -0.0141)
Holt Linear Trend Prediction:        25.2153 (Error = -0.5799)
ARIMA(0,1,0)+drift Prediction:       25.2211 (Error = -0.5856)
Quadratic Trend Prediction:          25.5932 (Error = -0.9577)

=== 2030 EXTRAPOLATION COMPARISON ===
Linear Trend 2030:    10^26.98 = 9.57e+26 FLOP (Physically plausible)
Holt Linear 2030:     10^27.67 = 4.66e+27 FLOP (Dynamic trend)
ARIMA(0,1,0) 2030:    10^27.75 = 5.68e+27 FLOP (Differenced drift)
Quadratic Trend 2030: 10^29.48 = 3.01e+29 FLOP (Explosive/implausible)


### 7. Task 10: Module VII: Revised Hypothesis Testing
We partition the 16-year modern modeling horizon into two equal 8-year sub-periods (2010–2017 vs 2018–2025) to test for a central location shift:
$$H_0: \mu_1 = \mu_2 \quad \text{vs.} \quad H_1: \mu_1 \neq \mu_2 \quad (\alpha = 0.05)$$

*Methodological Note: This test evaluates temporal location shift across two equal halves of the modern era. It is an observational contrast, not a controlled quasi-experiment of any single technology.*

In [14]:
p1_vals = train_df[train_df['Year'] <= 2017]['median_log10_compute'].values
p2_vals = train_df[train_df['Year'] >= 2018]['median_log10_compute'].values

n1, n2 = len(p1_vals), len(p2_vals)
mean1, mean2 = np.mean(p1_vals), np.mean(p2_vals)
var1, var2 = np.var(p1_vals, ddof=1), np.var(p2_vals, ddof=1)

tt_stat, tt_pval = stats.ttest_ind(p1_vals, p2_vals, equal_var=False)
dof_w = (var1/n1 + var2/n2)**2 / ((var1/n1)**2 / (n1 - 1) + (var2/n2)**2 / (n2 - 1))
diff_m = mean2 - mean1
se_d = np.sqrt(var1/n1 + var2/n2)
t_crit_w = stats.t.ppf(0.975, df=dof_w)
ci_d = [diff_m - t_crit_w * se_d, diff_m + t_crit_w * se_d]

print('=== MODULE VII: HYPOTHESIS TESTING (WELCH T-TEST) ===')
print(f'Period 1 (2010–2017): n1 = {n1}, Mean = {mean1:.4f}, Var = {var1:.4f}')
print(f'Period 2 (2018–2025): n2 = {n2}, Mean = {mean2:.4f}, Var = {var2:.4f}')
print(f'Mean Difference:      {diff_m:.4f} log10(FLOP)')
print(f'Welch t-statistic:    {tt_stat:.4f}')
print(f'Degrees of Freedom:   {dof_w:.2f}')
print(f'p-value:              {tt_pval:.4e}')
print(f'95% CI for Diff:      [{ci_d[0]:.4f}, {ci_d[1]:.4f}]')
decision = 'Reject H0 (Statistically Significant Location Shift)' if tt_pval < 0.05 else 'Fail to Reject H0'
print(f'Decision (alpha=0.05): {decision}')

=== MODULE VII: HYPOTHESIS TESTING (WELCH T-TEST) ===
Period 1 (2010–2017): n1 = 8, Mean = 17.5000, Var = 1.7573
Period 2 (2018–2025): n2 = 8, Mean = 21.8920, Var = 4.7864
Mean Difference:      4.3919 log10(FLOP)
Welch t-statistic:    -4.8561
Degrees of Freedom:   11.53
p-value:              4.4197e-04
95% CI for Diff:      [2.4124, 6.3715]
Decision (alpha=0.05): Reject H0 (Statistically Significant Location Shift)


### 8. Final Primary Forecasts (2027–2030) with Prediction Intervals
Based on the 7-fold backtest stability, unmatched 2026 snapshot accuracy, and stable extrapolation, **Linear Trend (OLS)** is selected as the primary forecasting model. We derive 95% prediction intervals:
$$s_{\text{pred}}(t_0) = s_e \sqrt{1 + \frac{1}{n} + \frac{(t_0 - \bar{t})^2}{\sum (t_i - \bar{t})^2}}$$

In [16]:
future_years = [2027, 2028, 2029, 2030]
x_bar = np.mean(x_train)
ss_xx = np.sum((x_train - x_bar)**2)
res_lin = y_train - (int_full + s_full * x_train)
se_res_lin = np.sqrt(np.sum(res_lin**2) / (n_train - 2))
t_crit_95 = stats.t.ppf(0.975, df=n_train - 2)

final_fc_rows = []
for yf in future_years:
    y_pred_lin = int_full + s_full * yf
    se_pred = se_res_lin * np.sqrt(1 + 1/n_train + (yf - x_bar)**2 / ss_xx)
    pi_low = y_pred_lin - t_crit_95 * se_pred
    pi_high = y_pred_lin + t_crit_95 * se_pred
    
    final_fc_rows.append({
        'Year': yf,
        'Predicted_log10_FLOP': round(y_pred_lin, 4),
        'PI_Lower_95_log10': round(pi_low, 4),
        'PI_Upper_95_log10': round(pi_high, 4),
        'Predicted_FLOP': 10**y_pred_lin,
        'PI_Lower_95_FLOP': 10**pi_low,
        'PI_Upper_95_FLOP': 10**pi_high
    })

final_fc_df = pd.DataFrame(final_fc_rows)
out_ffc = 'outputs/final_forecast_2027_2030.csv' if os.path.exists('outputs') else '../outputs/final_forecast_2027_2030.csv'
final_fc_df.to_csv(out_ffc, index=False)
print('=== FINAL PRIMARY FORECASTS (2027–2030) ===')
final_fc_df[['Year', 'Predicted_log10_FLOP', 'PI_Lower_95_log10', 'PI_Upper_95_log10', 'Predicted_FLOP']]

=== FINAL PRIMARY FORECASTS (2027–2030) ===


### 9. Figure 16: Final Model Selection & Projections to 2030
Figure 16 displays historical observations, the 7-fold backtest horizon, the partial 2026 snapshot, the selected Linear Trend forecast with 95% prediction interval band, and competing model trajectories.

In [18]:
fig, ax = plt.subplots(figsize=(10.5, 5.5))

# Historical points
ax.scatter(x_train, y_train, color='#263238', s=65, edgecolor='black', zorder=5, label='Historical Annual Median (2010–2025)')
ax.axvspan(2018.5, 2025.5, color='#f1f8e9', alpha=0.45, label='7-Fold Rolling Backtest Horizon (2019–2025)')
ax.scatter([2026], [actual_2026], color='#f57c00', s=90, marker='D', edgecolor='black', zorder=6, label='2026 Partial Snapshot (Actual = 24.64)')

# Forecast vectors (2026-2030)
fc_years_plot = np.array([2026, 2027, 2028, 2029, 2030])
fc_lin_plot = int_full + s_full * fc_years_plot
se_pred_plot = se_res_lin * np.sqrt(1 + 1/n_train + (fc_years_plot - x_bar)**2 / ss_xx)
pi_low_plot = fc_lin_plot - t_crit_95 * se_pred_plot
pi_high_plot = fc_lin_plot + t_crit_95 * se_pred_plot

ax.plot(fc_years_plot, fc_lin_plot, color='#1565c0', marker='o', lw=2.4, label='Selected Primary Model: Linear Trend (OLS)')
ax.fill_between(fc_years_plot, pi_low_plot, pi_high_plot, color='#bbdefb', alpha=0.5, label='Linear 95% Prediction Interval')

# Competing model trajectories
ax.plot(fc_years_plot, ar010_full.forecast(5), color='#6a1b9a', marker='v', lw=1.6, linestyle=':', label='ARIMA(0,1,0)+drift (Best 7-Fold Backtest RMSE=0.824)')
ax.plot(fc_years_plot, holt_full.forecast(5), color='#e65100', marker='p', lw=1.6, linestyle='--', label='Holt Linear Trend (Exponential Smoothing)')
ax.plot(fc_years_plot, np.polyval(p2_full, fc_years_plot - 2010), color='#c62828', marker='^', lw=1.5, linestyle='-.', label='Quadratic Trend (Rejected: Explosive Acceleration)')

ax.set_title('Figure 16: Final Model Selection & Projections of AI Training Compute to 2030')
ax.set_xlabel('Year')
ax.set_ylabel('Median log10(Training Compute (FLOP))')
ax.set_ylim(14, 30.5)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(frameon=True, facecolor='white', framealpha=0.9, loc='upper left', fontsize=8.5)
plt.tight_layout()
plt.savefig('graphs/16_final_model_selection_and_forecast.png' if os.path.exists('graphs') else '../graphs/16_final_model_selection_and_forecast.png')
plt.close()
print('Saved Figure 16 successfully.')

Saved Figure 16 successfully.


### 10. Final Model Selection Console Output

In [20]:
print(f'''
FINAL MODEL SELECTION
---------------------
Selected model: Linear Trend (OLS)
Reason: Across the extended 7-fold rolling backtest (2019–2025), Linear Trend maintained steady out-of-sample accuracy (RMSE=0.8900, OOS R2=0.6818) and predicted the 2026 partial-year snapshot with unmatched precision (error of -0.0141). Unlike Quadratic regression which collapsed in the 7-fold backtest (RMSE=1.2088) and explodes to 10^29.5 FLOP, Linear Trend produces an orderly, physically defensible multi-year extrapolation with robust degrees of freedom (14 df on N=16).
Backtest MAE: 0.8023
Backtest RMSE: 0.8900
OOS R2: 0.6818
2026 partial-year error: -0.0141 (Predicted 24.6496 vs Actual 24.6355)
2030 forecast: 26.9807 log10 FLOP (9.57e+26 FLOP)
2030 prediction interval: [25.0453, 28.9161] log10 FLOP ([1.11e+25, 8.24e+28] FLOP)
''')


FINAL MODEL SELECTION
---------------------
Selected model: Linear Trend (OLS)
Reason: Across the extended 7-fold rolling backtest (2019–2025), Linear Trend maintained steady out-of-sample accuracy (RMSE=0.8900, OOS R2=0.6818) and predicted the 2026 partial-year snapshot with unmatched precision (error of -0.0141). Unlike Quadratic regression which collapsed in the 7-fold backtest (RMSE=1.2088) and explodes to 10^29.5 FLOP, Linear Trend produces an orderly, physically defensible multi-year extrapolation with robust degrees of freedom (14 df on N=16).
Backtest MAE: 0.8023
Backtest RMSE: 0.8900
OOS R2: 0.6818
2026 partial-year error: -0.0141 (Predicted 24.6496 vs Actual 24.6355)
2030 forecast: 26.9807 log10 FLOP (9.57e+26 FLOP)
2030 prediction interval: [25.0453, 28.9161] log10 FLOP ([1.11e+25, 8.24e+28] FLOP)

